# Multi-class PCOS Subtype Classification
**Authors:** Madhu Parameswari Ganesan, Mahalakshmi M, Dhivyalakshmi S

Proxy-rule PCOS phenotypes from public clinical data, a leakage-free XGBoost + neural-network ensemble with per-subtype SHAP,
an **audited** ultrasound CNN, and a SHAP-grounded GenAI advisory. Every number printed or exported is computed in the run.



In [ ]:
# ============================================================
#  Mount Drive and fix paths
# ============================================================
import os

if os.path.isdir('/content/drive/MyDrive'):
    print("Drive already mounted, skipping mount")
else:
    from google.colab import drive
    drive.mount('/content/drive')

print("-" * 40)
BASE_PATH   = '/content/drive/MyDrive/pcos_project'
OUD_PATH    = os.path.join(BASE_PATH, 'data', 'ovary_us')
OUTPUT_PATH = os.path.join(BASE_PATH, 'outputs')
for _sub in ['csv', 'plots', 'models']:
    os.makedirs(os.path.join(OUTPUT_PATH, _sub), exist_ok=True)

print("Base path exists   :", os.path.isdir(BASE_PATH))
print("Dataset path       :", OUD_PATH, os.path.isdir(OUD_PATH))
print("-" * 40)
print(os.listdir(OUD_PATH) if os.path.isdir(OUD_PATH) else "dataset folder not found")

In [ ]:
# ============================================================
#  Configuration
#  All paths, seeds, thresholds and model settings live here.
# ============================================================
import os, sys, json, random, shutil, warnings, subprocess, importlib
warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED)

IN_COLAB = False
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except Exception:
    pass

BASE_PATH = os.environ.get(
    'PCOS_BASE_PATH',
    '/content/drive/MyDrive/pcos_project' if IN_COLAB else './pcos_project')
TABULAR_FILE  = os.path.join(BASE_PATH, 'data', 'tabular', 'PCOS_data_without_infertility.xlsx')
TABULAR_SHEET = 'Full_new'
OUD_PATH      = os.path.join(BASE_PATH, 'data', 'ovary_us')        # Hamadan Ovarian Ultrasound Dataset: class folders PCO / Normal / Dominant Follicle
OUTPUT_PATH   = os.path.join(BASE_PATH, 'outputs')
for _sub in ['csv', 'plots', 'models']:
    os.makedirs(os.path.join(OUTPUT_PATH, _sub), exist_ok=True)

# ---- Proxy-rule thresholds (cascade order follows the PDF: IR -> Inflammatory -> Adrenal -> Post-pill)
RULES = {
    'bmi_ir'          : 25.0,   # BMI >= 25 (Asian-Indian overweight cut-off)
    'whr_ir'          : 0.85,   # waist:hip > 0.85 (WHO, women)
    'rbs_ir'          : 140.0,  # random blood sugar >= 140 mg/dL
    'ir_min_criteria' : 2,      # >= 2 of {skin darkening, BMI, WHR, RBS} -> insulin-resistant proxy
    'vitd_low'        : 20.0,   # Vit D3 < 20 ng/mL (PDF rule)
    'lh_fsh_high'     : 2.0,    # LH/FSH > 2 = elevated LH
}
SUBTYPES = ['Insulin_Resistant', 'Inflammatory', 'Adrenal', 'Post_Pill']
# Literature ranges quoted in the project PDF (percent of PCOS patients) -- for comparison only
PDF_REFERENCE_PCT = {'Insulin_Resistant': (65, 70), 'Inflammatory': (30, 35),
                     'Adrenal': (10, 15), 'Post_Pill': (15, 20)}
MIN_CLASS_SIZE = 10            # subtypes with fewer patients are excluded from modelling (and reported)
VITD_MAX_PLAUSIBLE = 150.0      # Vit D3 above this (ng/mL) is a data-entry error (the file has values near 5400-6000) -> set to missing; None disables

# Columns the rules read (rows missing any of these cannot be labelled)
LABEL_INPUTS = ['Skin_Darkening', 'BMI', 'Waist_Hip_Ratio', 'Blood_Sugar', 'VitD3',
                'Hair_Growth', 'Pimples', 'Hair_Loss', 'LH_FSH_Ratio']
# Removed in the "blind" experiment: rule inputs + the raw columns they are computed from
LEAKY_FOR_BLIND = LABEL_INPUTS + ['LH', 'FSH', 'FSH_LH_raw', 'Weight', 'Height', 'Hip', 'Waist']
META_EXCLUDE    = ['PCOS', 'Blood_Group']

# ---- Modelling
KNN_K            = 5
TOP_K_FEATURES   = 15          # PDF: top 15 SHAP features
N_SPLITS         = 5           # reduced automatically if the rarest subtype is smaller
BOOTSTRAP_N      = 1000
XGB_GRID         = {'max_depth': [3, 5], 'learning_rate': [0.05, 0.1], 'n_estimators': [100, 200]}
NN_EPOCHS        = 200
NN_PATIENCE      = 20
NN_BATCH         = 16
ENSEMBLE_W_XGB   = 0.5         # soft-voting weight of XGBoost (NN gets 1 - this)
FINAL_FEATURE_SET = 'full_features'   # which experiment's feature set the deployed model / SHAP / advisory use

# ---- Ultrasound CNN (Hamadan OUD; frozen ResNet50 features + logistic head, grouped cross-validation)
RUN_CNN        = True
IMG_SIZE       = (224, 224)
CNN_CLASSES    = ['Normal', 'PCO', 'Dominant_Follicle']   # 3-class task; order = class index 0, 1, 2
CNN_POSITIVE   = 'PCO'          # used only for Grad-CAM and the PCOS probability passed to the advisory
CNN_CV_FOLDS   = 5
CNN_LR_C       = 0.1            # fixed (not tuned on the evaluation folds)
N_PERM         = 20             # label-permutation runs (leakage check; should fall to chance)
CNN_WEIGHTS    = 'imagenet'     # use None only for offline smoke tests
MIN_IMAGES_PER_CLASS = 20

# ---- Optional fine-tuning of ResNet50 (needs a GPU; ~10-20 min on a Colab T4). Fixed settings: nothing is tuned on the test folds.
RUN_FINETUNE      = True
FT_ALLOW_CPU      = False       # True = run without a GPU (hours)
FT_HEAD_EPOCHS    = 5           # phase 1: new classification head only
FT_EPOCHS         = 12          # phase 2: last ResNet stage unfrozen
FT_LR             = 1e-5        # phase-2 learning rate
FT_BATCH          = 16
FT_UNFREEZE_FROM  = 'conv5_block1_1_conv'   # first layer of ResNet50's last stage
FT_ROTATION       = 0.04        # +-0.04 turn (about +-14 degrees)
FT_ZOOM           = 0.10
FT_CONTRAST       = 0.15

# ---- Image audit (Module 5)
NORMALIZE_RES      = False     # True = blur every image to LOW_RES x LOW_RES before upsampling (weakens size/sharpness cues)
CROP_SQUARE        = True      # centre-crop every image to a square before resizing (removes aspect-ratio / native-size cues)
EQUALIZE_INTENSITY = True      # per-image histogram equalisation (removes overall brightness / contrast differences between sources)
SIZE_SHORTCUT_COUNTS = False   # raw width/height is always reported; it only flags a shortcut if True (the CNN never sees raw size once images are cropped and resized)
LOW_RES            = 96
SHORTCUT_THRESHOLD = 0.90      # if width/height or simple pixel statistics reach this accuracy, the CNN score is flagged non-diagnostic
BORDER_FRAC        = 0.15      # Grad-CAM: width of the image margin treated as 'border'
N_GRADCAM          = 60        # images used for the border-attention measurement

# ---- GenAI advisory (key is read from Colab Secrets 'GROQ_API_KEY' or env var GROQ_API_KEY -- never typed here)
RUN_GENAI          = True
GROQ_MODEL         = os.environ.get('GROQ_MODEL', 'openai/gpt-oss-120b')
N_DEMO_PER_SUBTYPE = 1
ADVISORY_DISCLAIMER = ('Proxy-rule label from a small public dataset, not a diagnosis. '
                       'Educational information only - discuss with a qualified clinician.')

print('Configuration loaded.')
print(f'  Base path : {BASE_PATH}')
print(f'  In Colab  : {IN_COLAB}')

In [ ]:
# ============================================================
#  Libraries
# ============================================================
def _ensure(pkg, mod=None):
    try:
        importlib.import_module(mod or pkg)
    except ImportError:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg], check=False)

for _p, _m in [('xgboost', 'xgboost'), ('shap', 'shap'), ('openpyxl', 'openpyxl'), ('groq', 'groq')]:
    _ensure(_p, _m)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from scipy import stats

from sklearn.impute import KNNImputer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold, GridSearchCV, train_test_split
from sklearn.utils.class_weight import compute_sample_weight, compute_class_weight
from sklearn.metrics import (accuracy_score, f1_score, classification_report,
                             confusion_matrix, roc_auc_score)
import joblib, hashlib, io
from PIL import Image
from scipy.ndimage import laplace
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.ensemble import RandomForestClassifier
import shap
import tensorflow as tf
from xgboost import XGBClassifier

np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

plt.rcParams.update({'figure.dpi': 120, 'axes.titlesize': 11, 'axes.labelsize': 10})

def save_fig(name, dpi=200):
    plt.savefig(os.path.join(OUTPUT_PATH, 'plots', name), dpi=dpi, bbox_inches='tight')

print('Libraries imported.')
print(f'  pandas {pd.__version__} | numpy {np.__version__} | shap {shap.__version__} | tensorflow {tf.__version__}')
tf.get_logger().setLevel('ERROR')

## Module 1 — Data loading, cohort, feature engineering
Only the Kottarathil (Kaggle) clinical file is used for the tabular pipeline. The second tabular file used earlier has none of the columns the label rules need (skin darkening, waist:hip, Vit D3...), so after merging its rows would be empty for every rule input and add nothing but imputed noise.

In [ ]:
# ============================================================
#   Load clinical dataset (Kottarathil, Kaggle)
# ============================================================
RENAME_MAP = {
    'PCOS (Y/N)': 'PCOS', 'Age (yrs)': 'Age', 'Weight (Kg)': 'Weight', 'Height(Cm)': 'Height',
    'Blood Group': 'Blood_Group', 'Pulse rate(bpm)': 'Pulse_Rate', 'RR (breaths/min)': 'RR',
    'Hb(g/dl)': 'Hemoglobin', 'Cycle(R/I)': 'Cycle', 'Cycle length(days)': 'Cycle_Length',
    'Marraige Status (Yrs)': 'Marriage_Years', 'Pregnant(Y/N)': 'Pregnant',
    'No. of aborptions': 'Abortions', 'I   beta-HCG(mIU/mL)': 'HCG_I',
    'II    beta-HCG(mIU/mL)': 'HCG_II', 'FSH(mIU/mL)': 'FSH', 'LH(mIU/mL)': 'LH',
    'FSH/LH': 'FSH_LH_raw', 'Hip(inch)': 'Hip', 'Waist(inch)': 'Waist',
    'Waist:Hip Ratio': 'Waist_Hip_Ratio', 'TSH (mIU/L)': 'TSH', 'AMH(ng/mL)': 'AMH',
    'PRL(ng/mL)': 'Prolactin', 'Vit D3 (ng/mL)': 'VitD3', 'PRG(ng/mL)': 'Progesterone',
    'RBS(mg/dl)': 'Blood_Sugar', 'Weight gain(Y/N)': 'Weight_Gain', 'hair growth(Y/N)': 'Hair_Growth',
    'Skin darkening (Y/N)': 'Skin_Darkening', 'Hair loss(Y/N)': 'Hair_Loss', 'Pimples(Y/N)': 'Pimples',
    'Fast food (Y/N)': 'Fast_Food', 'Reg.Exercise(Y/N)': 'Exercise',
    'BP _Systolic (mmHg)': 'BP_Systolic', 'BP _Diastolic (mmHg)': 'BP_Diastolic',
    'Follicle No. (L)': 'Follicle_L', 'Follicle No. (R)': 'Follicle_R',
    'Avg. F size (L) (mm)': 'Follicle_Size_L', 'Avg. F size (R) (mm)': 'Follicle_Size_R',
    'Endometrium (mm)': 'Endometrium',
}

if not os.path.exists(TABULAR_FILE):
    raise FileNotFoundError(f'Clinical file not found: {TABULAR_FILE}\nSet PCOS_BASE_PATH or fix TABULAR_FILE in CELL 1.')

df_raw = pd.read_excel(TABULAR_FILE, sheet_name=TABULAR_SHEET)
df_raw.columns = df_raw.columns.astype(str).str.strip()
df_raw = df_raw.drop(columns=[c for c in df_raw.columns
                              if c in ('Sl. No', 'Patient File No.') or c.startswith('Unnamed')])
df_raw = df_raw.rename(columns={k: v for k, v in RENAME_MAP.items() if k in df_raw.columns})

# some cells in this Excel file hold stray text (e.g. '1.99.') -> coerce everything to numeric
for c in df_raw.columns:
    df_raw[c] = pd.to_numeric(df_raw[c], errors='coerce')

needed = ['PCOS'] + [c for c in LABEL_INPUTS if c not in ('LH_FSH_Ratio',)] + ['LH', 'FSH', 'Follicle_L', 'Follicle_R']
missing_cols = [c for c in needed if c not in df_raw.columns]
if missing_cols:
    raise KeyError(f'Required columns missing after renaming: {missing_cols}\nAvailable: {list(df_raw.columns)}')

print('Clinical dataset loaded')
print('-' * 40)
print(f'  Patients (all)  : {len(df_raw)}')
print(f'  Columns         : {df_raw.shape[1]}')
print(f'  PCOS positive   : {int((df_raw["PCOS"] == 1).sum())}')
print(f'  PCOS negative   : {int((df_raw["PCOS"] == 0).sum())}')

In [ ]:
# ============================================================
#  PCOS cohort + feature engineering (no imputation here)
#  Imputation happens later, inside each CV training fold.
# ============================================================
cohort = df_raw[df_raw['PCOS'] == 1].copy().reset_index(drop=True)

# physiologically impossible zero -> missing
cohort['VitD3'] = cohort['VitD3'].replace(0, np.nan)
n_vitd_bad = 0
if VITD_MAX_PLAUSIBLE is not None:
    _bad = cohort['VitD3'] > VITD_MAX_PLAUSIBLE
    n_vitd_bad = int(_bad.sum())
    cohort.loc[_bad, 'VitD3'] = np.nan

# fill derived columns only where the recorded value is absent
if {'Weight', 'Height'} <= set(cohort.columns):
    cohort['BMI'] = cohort['BMI'].fillna(cohort['Weight'] / ((cohort['Height'] / 100.0) ** 2)) \
        if 'BMI' in cohort.columns else cohort['Weight'] / ((cohort['Height'] / 100.0) ** 2)
if {'Waist', 'Hip'} <= set(cohort.columns):
    cohort['Waist_Hip_Ratio'] = cohort['Waist_Hip_Ratio'].fillna(cohort['Waist'] / cohort['Hip'])

# engineered features
cohort['LH_FSH_Ratio']    = cohort['LH'] / cohort['FSH'].where(cohort['FSH'] > 0)
cohort['Total_Follicles'] = cohort[['Follicle_L', 'Follicle_R']].sum(axis=1, min_count=2)
size_cols = [c for c in ['Follicle_Size_L', 'Follicle_Size_R'] if c in cohort.columns]
if size_cols:
    cohort['Avg_Follicle_Size'] = cohort[size_cols].mean(axis=1)

# HOMA-IR = fasting insulin (uIU/mL) x fasting glucose (mg/dL) / 405  -> only if both truly exist
if {'Fasting_Insulin', 'Fasting_Glucose'} <= set(cohort.columns):
    cohort['HOMA_IR'] = cohort['Fasting_Insulin'] * cohort['Fasting_Glucose'] / 405.0
    homa_status = 'computed from fasting insulin x fasting glucose / 405'
else:
    homa_status = 'NOT computed: this dataset has no fasting insulin (random blood sugar alone is not HOMA-IR)'

print('Cohort + features')
print('-' * 40)
print(f'  PCOS cohort size : {len(cohort)}')
print(f'  HOMA-IR          : {homa_status}')
print(f'  Vit D3 > {VITD_MAX_PLAUSIBLE} set to missing : {n_vitd_bad}')
print('\nMissing values in rule-input columns:')
print(cohort[LABEL_INPUTS].isnull().sum().to_string())

## Module 2 — Rule-based subtype labels
Cascade using the columns that exist in the data:

| Order | Subtype | Proxy rule |
|---|---|---|
| 1 | Insulin_Resistant | at least `ir_min_criteria` of: skin darkening (acanthosis), BMI ≥ `bmi_ir`, waist:hip > `whr_ir`, RBS ≥ `rbs_ir` |
| 2 | Inflammatory | Vit D3 < `vitd_low` *(CRP and WBC are not in the dataset)* |
| 3 | Adrenal | hirsutism, acne or hair loss **and** LH/FSH ≤ `lh_fsh_high` *(DHEA-S / testosterone not in the dataset)* |
| 4 | Post_Pill | LH/FSH > `lh_fsh_high` and no insulin-resistance criteria *(OCP history not in the dataset)* |

Patients matching none of the rules are reported as *Unclassified* and left out of training instead of being forced into a class.

In [ ]:
# ============================================================
#  Assign subtype labels from the rules
# ============================================================
def assign_subtypes(d, R):
    ir_flags = pd.DataFrame({
        'skin': d['Skin_Darkening'] == 1,
        'bmi' : d['BMI'] >= R['bmi_ir'],
        'whr' : d['Waist_Hip_Ratio'] > R['whr_ir'],
        'rbs' : d['Blood_Sugar'] >= R['rbs_ir'],
    })
    ir_count  = ir_flags.sum(axis=1)
    is_ir     = ir_count >= R['ir_min_criteria']
    is_infl   = d['VitD3'] < R['vitd_low']
    androgen  = (d['Hair_Growth'] == 1) | (d['Pimples'] == 1) | (d['Hair_Loss'] == 1)
    is_adr    = androgen & (d['LH_FSH_Ratio'] <= R['lh_fsh_high'])
    is_pp     = (d['LH_FSH_Ratio'] > R['lh_fsh_high']) & (ir_count == 0)
    labels = np.select([is_ir, is_infl, is_adr, is_pp],
                       ['Insulin_Resistant', 'Inflammatory', 'Adrenal', 'Post_Pill'],
                       default='Unclassified')
    return pd.Series(labels, index=d.index, name='Subtype_Name')

n_before  = len(cohort)
labelable = cohort.dropna(subset=LABEL_INPUTS).copy()
n_unlabelable = n_before - len(labelable)
labelable['Subtype_Name'] = assign_subtypes(labelable, RULES)

counts_all = labelable['Subtype_Name'].value_counts()
n_unclassified = int(counts_all.get('Unclassified', 0))

kept_classes = [s for s in SUBTYPES if counts_all.get(s, 0) >= MIN_CLASS_SIZE]
dropped_small = {s: int(counts_all.get(s, 0)) for s in SUBTYPES if s not in kept_classes}
if len(kept_classes) < 2:
    raise RuntimeError(f'Fewer than 2 subtypes reach MIN_CLASS_SIZE={MIN_CLASS_SIZE}: {counts_all.to_dict()}. '
                       'Adjust RULES or MIN_CLASS_SIZE in CELL 1.')

df_model    = labelable[labelable['Subtype_Name'].isin(kept_classes)].copy().reset_index(drop=True)
class_names = kept_classes
n_classes   = len(class_names)
label_map   = {n: i for i, n in enumerate(class_names)}
y           = df_model['Subtype_Name'].map(label_map).astype(int).values

# feature matrix: numeric columns only, label & meta columns can never enter
numeric_cols = df_model.select_dtypes(include=[np.number]).columns
feature_cols = [c for c in numeric_cols
                if c not in META_EXCLUDE
                and df_model[c].notna().any()
                and df_model[c].nunique(dropna=True) > 1]
X_all = df_model[feature_cols].astype(float)
assert not any(c.lower().startswith('subtype') for c in X_all.columns), 'label leaked into features'

print('Subtype labelling (rule-based proxies)')
print('-' * 55)
print(f'  PCOS patients                    : {n_before}')
print(f'  Missing a rule input (excluded)  : {n_unlabelable}')
print(f'  Unclassified by every rule       : {n_unclassified}')
print(f'  Used for modelling               : {len(df_model)}')
if dropped_small:
    print(f'  Dropped (< {MIN_CLASS_SIZE} patients)        : {dropped_small}')
print('-' * 55)
for s in class_names:
    c = int((df_model["Subtype_Name"] == s).sum())
    print(f'  {s:<20}: {c:4d}  ({c / len(df_model) * 100:5.1f}%) ' + chr(9608) * int(c / len(df_model) * 40))
print(f'\n  Candidate features : {X_all.shape[1]}')
df_model.assign(Subtype_Code=y).to_csv(os.path.join(OUTPUT_PATH, 'csv', 'pcos_labeled_cohort.csv'), index=False)

In [ ]:
# ============================================================
#  Label validation (all values computed)
#  (a) distribution vs the ranges quoted in the PDF
#  (b) do subtypes differ on columns the rules did NOT use?
#  NOTE: this is a sanity check, not clinical validation.
# ============================================================
dist = df_model['Subtype_Name'].value_counts().reindex(class_names)
dist_df = pd.DataFrame({'Count': dist, 'Percent': (dist / dist.sum() * 100).round(1)})
dist_df['PDF_Reference_Percent'] = [f'{PDF_REFERENCE_PCT[s][0]}-{PDF_REFERENCE_PCT[s][1]}' for s in dist_df.index]
dist_df['Within_PDF_Range'] = [PDF_REFERENCE_PCT[s][0] <= p <= PDF_REFERENCE_PCT[s][1]
                               for s, p in dist_df['Percent'].items()]
dist_df.to_csv(os.path.join(OUTPUT_PATH, 'csv', 'label_distribution.csv'))
print('(a) Distribution vs PDF reference ranges')
print('-' * 70)
print(dist_df.to_string())
print('(PDF ranges are literature figures for biomarker-defined subtypes and overlap; '
      'a mismatch is expected with proxy rules and should be discussed, not hidden.)')

def bh_fdr(p):
    p = np.asarray(p, float)
    n = len(p)
    order = np.argsort(p)
    ranked = p[order] * n / np.arange(1, n + 1)
    ranked = np.minimum.accumulate(ranked[::-1])[::-1]
    out = np.empty(n)
    out[order] = np.minimum(ranked, 1.0)
    return out

test_cols = [c for c in X_all.columns if c not in LEAKY_FOR_BLIND]
rows = []
for c in test_cols:
    groups = [df_model.loc[df_model['Subtype_Name'] == s, c].dropna().values for s in class_names]
    groups = [g for g in groups if len(g) >= 2]
    if len(groups) < 2:
        continue
    pooled = np.concatenate(groups)
    if np.allclose(pooled, pooled[0]):
        continue
    try:
        h, p = stats.kruskal(*groups)
    except ValueError:
        continue
    rows.append({'Feature': c, 'Kruskal_H': h, 'p_value': p})

kw = pd.DataFrame(rows)
if len(kw):
    kw['p_FDR_BH'] = bh_fdr(kw['p_value'].values)
    kw = kw.sort_values('p_value').reset_index(drop=True)
    kw.to_csv(os.path.join(OUTPUT_PATH, 'csv', 'label_validation_kruskal.csv'), index=False)
    n_sig = int((kw['p_FDR_BH'] < 0.05).sum())
    print(f'\n(b) Kruskal-Wallis on {len(kw)} columns NOT used by the rules: '
          f'{n_sig} differ between subtypes at FDR < 0.05')
    print(kw.head(10).round(4).to_string(index=False))
else:
    n_sig = 0
    print('\n(b) No testable columns.')


In [ ]:
# ============================================================
#   EDA plots
# ============================================================
fig, ax = plt.subplots(figsize=(6, 4))
cnt = df_model['Subtype_Name'].value_counts().reindex(class_names)
bars = ax.bar(cnt.index, cnt.values)
for b, v in zip(bars, cnt.values):
    ax.text(b.get_x() + b.get_width() / 2, v, f'{v}\n({v / cnt.sum() * 100:.1f}%)', ha='center', va='bottom', fontsize=9)
ax.set_title('Figure 1. Subtype class distribution')
ax.set_ylabel('Patients')
ax.set_ylim(0, cnt.max() * 1.2)
plt.xticks(rotation=15)
plt.tight_layout(); save_fig('Fig1_class_distribution.png'); plt.show()

eda_feats = [c for c in ['BMI', 'Waist_Hip_Ratio', 'Blood_Sugar', 'VitD3', 'LH_FSH_Ratio',
                         'AMH', 'Prolactin', 'TSH', 'Total_Follicles', 'Age'] if c in df_model.columns]
ncol = 5
nrow = int(np.ceil(len(eda_feats) / ncol))
fig, axes = plt.subplots(nrow, ncol, figsize=(4 * ncol, 3.4 * nrow))
for ax, f in zip(np.atleast_1d(axes).ravel(), eda_feats):
    sns.boxplot(data=df_model, x='Subtype_Name', y=f, order=class_names, ax=ax)
    ax.set_xlabel(''); ax.set_title(f)
    ax.tick_params(axis='x', rotation=30)
for ax in np.atleast_1d(axes).ravel()[len(eda_feats):]:
    ax.axis('off')
fig.suptitle('Figure 2. Key clinical features by subtype', y=1.02)
plt.tight_layout(); save_fig('Fig2_feature_boxplots.png'); plt.show()

fig, ax = plt.subplots(figsize=(8, 6.5))
corr = df_model[eda_feats].corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0, ax=ax)
ax.set_title('Figure 3. Correlation of key features')
plt.tight_layout(); save_fig('Fig3_correlation.png'); plt.show()

## Module 3 — Models (XGBoost, Neural Network, Ensemble) with leak-free cross-validation
For every outer fold: impute (KNN, k=5) → SHAP top-K selection → tune XGBoost (inner CV) → train NN → soft-vote ensemble. Everything is fitted on the training part of that fold only.

In [ ]:
# ============================================================
#   Helper functions
# ============================================================
def make_imputer():
    try:
        return KNNImputer(n_neighbors=KNN_K, weights='distance', keep_empty_features=True)
    except TypeError:
        return KNNImputer(n_neighbors=KNN_K, weights='distance')

def impute_df(imp, X):
    return pd.DataFrame(imp.transform(X), columns=X.columns, index=X.index)

def shap_per_class(model, X, n_cls):
    # returns array (n_classes, n_samples, n_features) for any SHAP version
    sv = shap.TreeExplainer(model).shap_values(X)
    if isinstance(sv, list):
        return np.stack([np.asarray(s) for s in sv], axis=0)
    sv = np.asarray(sv)
    if sv.ndim == 3:
        return np.transpose(sv, (2, 0, 1)) if sv.shape[2] == n_cls else sv
    return np.stack([-sv, sv], axis=0)          # binary case

def select_top_k(X_imp, y_tr, k):
    k = min(k, X_imp.shape[1])
    sel = XGBClassifier(n_estimators=150, max_depth=3, learning_rate=0.1, subsample=0.8,
                        colsample_bytree=0.8, random_state=SEED, n_jobs=-1)
    sel.fit(X_imp, y_tr, sample_weight=compute_sample_weight('balanced', y_tr))
    arr = shap_per_class(sel, X_imp, len(np.unique(y_tr)))
    importance = np.abs(arr).mean(axis=(0, 1))
    order = np.argsort(-importance)[:k]
    return list(X_imp.columns[order])

def fit_xgb(X_tr, y_tr):
    sw = compute_sample_weight('balanced', y_tr)
    inner_cv = int(min(3, np.bincount(y_tr).min()))
    base = XGBClassifier(random_state=SEED, n_jobs=-1)
    if inner_cv >= 2:
        gs = GridSearchCV(base, XGB_GRID, scoring='f1_macro',
                          cv=StratifiedKFold(inner_cv, shuffle=True, random_state=SEED), n_jobs=1)
        gs.fit(X_tr, y_tr, sample_weight=sw)
        return gs.best_estimator_, gs.best_params_
    base.fit(X_tr, y_tr, sample_weight=sw)
    return base, {}

def nn_predict(model, X):
    # direct call avoids the tf.function re-tracing warning that model.predict() causes inside CV loops
    return model(np.asarray(X, dtype='float32'), training=False).numpy()

def build_nn(n_features, n_cls):
    return tf.keras.Sequential([
        tf.keras.layers.Input(shape=(n_features,)),
        tf.keras.layers.Dense(64, activation='relu'),
        tf.keras.layers.Dropout(0.3),
        tf.keras.layers.Dense(32, activation='relu'),
        tf.keras.layers.Dropout(0.2),
        tf.keras.layers.Dense(n_cls, activation='softmax'),
    ])

def fit_nn(Xs_tr, y_tr, n_cls):
    tf.keras.utils.set_random_seed(SEED)
    cw = dict(enumerate(compute_class_weight('balanced', classes=np.arange(n_cls), y=y_tr)))
    try:
        Xa, Xv, ya, yv = train_test_split(Xs_tr, y_tr, test_size=0.15, stratify=y_tr, random_state=SEED)
    except ValueError:
        Xa, Xv, ya, yv = train_test_split(Xs_tr, y_tr, test_size=0.15, random_state=SEED)
    model = build_nn(Xs_tr.shape[1], n_cls)
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    model.fit(Xa, ya, validation_data=(Xv, yv), epochs=NN_EPOCHS, batch_size=NN_BATCH,
              class_weight=cw, verbose=0,
              callbacks=[tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=NN_PATIENCE,
                                                          restore_best_weights=True)])
    return model

def score_preds(y_true, y_pred):
    f1_c = f1_score(y_true, y_pred, labels=list(range(n_classes)), average=None, zero_division=0)
    out = {'Accuracy'   : accuracy_score(y_true, y_pred),
           'Macro_F1'   : f1_score(y_true, y_pred, average='macro', zero_division=0),
           'Weighted_F1': f1_score(y_true, y_pred, average='weighted', zero_division=0)}
    for s, f in zip(class_names, f1_c):
        out[f'F1_{s}'] = f
    return out

def bootstrap_macro_f1_ci(y_true, y_pred, n_boot=BOOTSTRAP_N, seed=SEED):
    rng = np.random.default_rng(seed)
    vals = []
    for _ in range(n_boot):
        idx = rng.integers(0, len(y_true), len(y_true))
        vals.append(f1_score(y_true[idx], y_pred[idx], average='macro', zero_division=0))
    return float(np.percentile(vals, 2.5)), float(np.percentile(vals, 97.5))

def run_cv(X, y_vec):
    counts = np.bincount(y_vec, minlength=n_classes)
    n_splits = int(min(N_SPLITS, counts.min()))
    if n_splits < 2:
        raise ValueError('Rarest class has fewer than 2 patients.')
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=SEED)
    oof = {m: np.zeros((len(y_vec), n_classes)) for m in ['XGBoost', 'Neural_Network', 'Ensemble']}
    fold_acc = {m: [] for m in oof}
    selected, params = [], []
    for fold, (tr, te) in enumerate(skf.split(X, y_vec), 1):
        imp = make_imputer().fit(X.iloc[tr])
        X_tr = impute_df(imp, X.iloc[tr])
        X_te = impute_df(imp, X.iloc[te])
        feats = select_top_k(X_tr, y_vec[tr], TOP_K_FEATURES)
        selected.append(feats)

        xgb_m, bp = fit_xgb(X_tr[feats], y_vec[tr])
        params.append(bp)
        scaler = StandardScaler().fit(X_tr[feats])
        nn_m = fit_nn(scaler.transform(X_tr[feats]), y_vec[tr], n_classes)

        p_x = xgb_m.predict_proba(X_te[feats])
        p_n = nn_predict(nn_m, scaler.transform(X_te[feats]))
        p_e = ENSEMBLE_W_XGB * p_x + (1.0 - ENSEMBLE_W_XGB) * p_n
        for name, p in [('XGBoost', p_x), ('Neural_Network', p_n), ('Ensemble', p_e)]:
            oof[name][te] = p
            fold_acc[name].append(accuracy_score(y_vec[te], p.argmax(1)))
        tf.keras.backend.clear_session()
        print(f'  fold {fold}/{n_splits}: acc  XGB {fold_acc["XGBoost"][-1]:.3f} | '
              f'NN {fold_acc["Neural_Network"][-1]:.3f} | Ens {fold_acc["Ensemble"][-1]:.3f}')
    return {'oof': oof, 'fold_acc': fold_acc, 'selected': selected, 'params': params, 'n_splits': n_splits}

def summarize(res, y_vec):
    rows = {}
    for m, p in res['oof'].items():
        pred = p.argmax(1)
        sc = score_preds(y_vec, pred)
        lo, hi = bootstrap_macro_f1_ci(y_vec, pred)
        sc['Macro_F1_CI95_low']  = lo
        sc['Macro_F1_CI95_high'] = hi
        sc['Fold_Acc_Mean'] = float(np.mean(res['fold_acc'][m]))
        sc['Fold_Acc_Std']  = float(np.std(res['fold_acc'][m]))
        rows[m] = sc
    return pd.DataFrame(rows).T

def plot_confusion(y_true, y_pred, title, fname):
    cm = confusion_matrix(y_true, y_pred, labels=list(range(n_classes)))
    fig, ax = plt.subplots(figsize=(6.5, 5.2))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', linewidths=0.5, ax=ax,
                xticklabels=class_names, yticklabels=class_names)
    ax.set_xlabel('Predicted subtype'); ax.set_ylabel('Actual (rule-defined) subtype'); ax.set_title(title)
    plt.xticks(rotation=20, ha='right'); plt.tight_layout(); save_fig(fname); plt.show()

print('Helpers defined.')

In [ ]:
# ============================================================
#  Run both experiments (out-of-fold evaluation)
#  full_features  : every candidate feature (can re-learn the rules)
#  blind_features : rule-defining columns removed (needs other signal)
# ============================================================
feats_full  = list(X_all.columns)
feats_blind = [c for c in X_all.columns if c not in LEAKY_FOR_BLIND]
EXPERIMENTS = {'full_features': feats_full, 'blind_features': feats_blind}

majority_baseline = float(np.bincount(y).max() / len(y))
results, tables, reports = {}, {}, {}

for tag, cols in EXPERIMENTS.items():
    print('=' * 70)
    print(f'EXPERIMENT: {tag}  ({len(cols)} candidate features)')
    print('=' * 70)
    if len(cols) < 2:
        print('  skipped: too few features')
        continue
    res = run_cv(X_all[cols], y)
    results[tag] = res
    tbl = summarize(res, y)
    tables[tag] = tbl
    tbl.round(4).to_csv(os.path.join(OUTPUT_PATH, 'csv', f'model_comparison_{tag}.csv'))

    print(f'\nOut-of-fold results ({res["n_splits"]}-fold, every patient predicted once)')
    print('-' * 70)
    print(tbl[['Accuracy', 'Macro_F1', 'Weighted_F1', 'Fold_Acc_Mean', 'Fold_Acc_Std']].round(4).to_string())
    print(f'  Majority-class baseline accuracy: {majority_baseline:.4f}')

    ens_pred = res['oof']['Ensemble'].argmax(1)
    rep = classification_report(y, ens_pred, labels=list(range(n_classes)), target_names=class_names,
                                zero_division=0, output_dict=True)
    reports[tag] = pd.DataFrame(rep).T
    pd.DataFrame(rep).T.round(4).to_csv(os.path.join(OUTPUT_PATH, 'csv', f'ensemble_classification_report_{tag}.csv'))
    print('\nEnsemble per-subtype report')
    print(classification_report(y, ens_pred, labels=list(range(n_classes)), target_names=class_names, zero_division=0))
    plot_confusion(y, ens_pred, f'Ensemble confusion matrix ({tag}, out-of-fold)', f'ConfMat_Ensemble_{tag}.png')

    stab = pd.Series(Counter(f for fs in res['selected'] for f in fs)).sort_values(ascending=False)
    stab = (stab / res['n_splits']).rename('Fraction_of_folds_selected').to_frame()
    stab.to_csv(os.path.join(OUTPUT_PATH, 'csv', f'feature_selection_stability_{tag}.csv'))
    print(f'Most stable SHAP-selected features: {list(stab.index[:5])}')

if FINAL_FEATURE_SET not in results:
    raise RuntimeError(f'FINAL_FEATURE_SET={FINAL_FEATURE_SET!r} did not run. Choose one of {list(results)}.')

## Module 4 — Final model, subtype-specific SHAP, patient-level explanation
The final model is fitted on all labelled patients **only for explanation and deployment**; the reported performance above comes from the out-of-fold runs.

In [ ]:
# ============================================================
#   Final models + per-subtype SHAP + predict_patient()
# ============================================================
cnn_model, pcos_idx, resnet_preprocess, cnn_metrics = None, None, None, None   # filled in by the CNN module

X_final = X_all[EXPERIMENTS[FINAL_FEATURE_SET]]
FEATURE_COLUMNS = list(X_final.columns)

imp_final   = make_imputer().fit(X_final)
X_final_imp = impute_df(imp_final, X_final)
feats_final = select_top_k(X_final_imp, y, TOP_K_FEATURES)
xgb_final, params_final = fit_xgb(X_final_imp[feats_final], y)
scaler_final = StandardScaler().fit(X_final_imp[feats_final])
nn_final = fit_nn(scaler_final.transform(X_final_imp[feats_final]), y, n_classes)

joblib.dump({'imputer': imp_final, 'scaler': scaler_final, 'features': feats_final,
             'columns': FEATURE_COLUMNS, 'class_names': class_names, 'xgb': xgb_final,
             'feature_set': FINAL_FEATURE_SET, 'rules': RULES},
            os.path.join(OUTPUT_PATH, 'models', 'tabular_bundle.pkl'))
nn_final.save(os.path.join(OUTPUT_PATH, 'models', 'nn_model.keras'))

# ---- subtype-specific SHAP
shap_arr = shap_per_class(xgb_final, X_final_imp[feats_final], n_classes)    # (C, N, F)

global_imp = pd.Series(np.abs(shap_arr).mean(axis=(0, 1)), index=feats_final).sort_values(ascending=False)
global_imp.rename('Mean_abs_SHAP').to_csv(os.path.join(OUTPUT_PATH, 'csv', 'shap_global_importance.csv'))

top5_rows = []
fig, axes = plt.subplots(1, n_classes, figsize=(4.6 * n_classes, 4.4))
axes = np.atleast_1d(axes)
for c, name in enumerate(class_names):
    imp_c = pd.Series(np.abs(shap_arr[c]).mean(axis=0), index=feats_final).sort_values(ascending=False)
    top5 = imp_c.head(5)
    for rank, (f, v) in enumerate(top5.items(), 1):
        top5_rows.append({'Subtype': name, 'Rank': rank, 'Feature': f, 'Mean_abs_SHAP': float(v)})
    axes[c].barh(top5.index[::-1], top5.values[::-1])
    axes[c].set_title(name)
    axes[c].set_xlabel('mean |SHAP|')
fig.suptitle('Figure 4. Top-5 SHAP features per subtype', y=1.03)
plt.tight_layout(); save_fig('Fig4_SHAP_top5_per_subtype.png'); plt.show()

top5_df = pd.DataFrame(top5_rows)
top5_df.to_csv(os.path.join(OUTPUT_PATH, 'csv', 'shap_top5_per_subtype.csv'), index=False)
print('Top-5 SHAP features per subtype')
print('-' * 55)
for name in class_names:
    sub = top5_df[top5_df['Subtype'] == name]
    print(f'  {name:<20}: ' + ', '.join(sub['Feature']))

for c, name in enumerate(class_names):
    plt.figure()
    shap.summary_plot(shap_arr[c], X_final_imp[feats_final], show=False, max_display=10)
    plt.title(f'SHAP summary - {name}')
    save_fig(f'Fig4b_SHAP_summary_{name}.png'); plt.show()

# ---- patient-level prediction + explanation (+ optional ultrasound image: decision-level fusion)
def predict_patient(patient, image_path=None, top_n=3):
    row = pd.DataFrame([patient]).reindex(columns=FEATURE_COLUMNS).astype(float)
    row = impute_df(imp_final, row)
    p_x = xgb_final.predict_proba(row[feats_final])[0]
    p_n = nn_predict(nn_final, scaler_final.transform(row[feats_final]))[0]
    p_e = ENSEMBLE_W_XGB * p_x + (1.0 - ENSEMBLE_W_XGB) * p_n
    c = int(p_e.argmax())
    sv = shap_per_class(xgb_final, row[feats_final], n_classes)[c, 0, :]
    order = np.argsort(-np.abs(sv))[:top_n]
    drivers = [{'feature': feats_final[i], 'value': float(row[feats_final].iloc[0, i]),
                'shap': float(sv[i]),
                'direction': 'raises' if sv[i] > 0 else 'lowers'} for i in order]
    out = {'subtype': class_names[c], 'confidence': float(p_e[c]),
           'probabilities': {n: float(p) for n, p in zip(class_names, p_e)},
           'drivers': drivers, 'cnn_pcos_probability': None}
    if image_path and cnn_model is not None:
        img = load_cnn_image(image_path)[None]          # same normalisation as the training images
        probs = cnn_model(resnet_preprocess(img), training=False).numpy()[0]
        out['cnn_pcos_probability'] = float(probs[pcos_idx])
        out['cnn_class_probabilities'] = {c: float(p) for c, p in zip(image_audit['class_list'], probs)}
    return out

print('\nFinal models trained and saved; predict_patient() ready.')

## Module 5 — Ultrasound images: audit first, then CNN (Hamadan Ovarian Ultrasound Dataset)
The earlier Kaggle image set was unusable: every test file was a copy of a training file, and image size or simple pixel statistics alone classified it with 100% accuracy. This module therefore uses the **Hamadan Ovarian Ultrasound Dataset (OUD, CC BY 4.0)**, classes *PCO*, *Normal*, *Dominant Follicle*. Download it and place the class folders under `data/ovary_us` in your project folder.

The images come from **different patients** than the clinical table, so this is a separate detector, combined with the tabular model only at decision level. The notebook **audits the data before training**:
1. exact duplicates (MD5) and near-duplicate groups (perceptual hash); near-duplicates are never split across cross-validation folds;
2. for **every pair of classes**: how well **image width/height alone** and **simple pixel statistics** (brightness, contrast, sharpness, border brightness, histogram; no CNN) predict the class;
3. the CNN task is the full **3-class problem** (`CNN_CLASSES`: Normal / PCO / Dominant_Follicle); the pairwise audit is still printed and exported, and a 3-class shortcut baseline is computed.

The CNN is a **frozen ResNet50 + multinomial logistic head** evaluated by **grouped 5-fold cross-validation**, with a **label-permutation check**. If size/pixel statistics reach `SHORTCUT_THRESHOLD`, or the CNN does not beat them, the result is labelled accordingly. Without patient IDs the split is still image-level, so the score is not clinically validated.


In [ ]:
# ============================================================
#  Ultrasound image dataset: load -> audit (every class pair) -> choose the CNN task
#  Dataset: Hamadan Ovarian Ultrasound Dataset (OUD): PCO / Normal / Dominant Follicle
#  Put the downloaded folder at OUD_PATH (see Config). Class = folder name.
#  Nothing is deleted or moved; the dataset folder is only read.
# ============================================================
import itertools
IMG_EXT = ('.png', '.jpg', '.jpeg', '.bmp', '.tif', '.tiff')
image_audit, img_df = None, None
cnn_ready = bool(RUN_CNN and os.path.isdir(OUD_PATH))

def oud_class(path):
    parts = [p.lower() for p in os.path.relpath(path, OUD_PATH).split(os.sep)[:-1]]
    for p in reversed(parts):
        if 'dominant' in p: return 'Dominant_Follicle'
        if 'normal' in p:   return 'Normal'
        if 'pco' in p:      return 'PCO'
    return None

def to_model_gray(im):
    """The exact grayscale representation the CNN sees (also used for the pixel-statistics audit)."""
    im = im.convert('L')
    if CROP_SQUARE:
        w_, h_ = im.size; m_ = min(w_, h_)
        im = im.crop(((w_ - m_) // 2, (h_ - m_) // 2, (w_ - m_) // 2 + m_, (h_ - m_) // 2 + m_))
    if NORMALIZE_RES:
        im = im.resize((LOW_RES, LOW_RES), Image.LANCZOS)
    im = im.resize(IMG_SIZE, Image.BICUBIC)
    if EQUALIZE_INTENSITY:
        from PIL import ImageOps
        im = ImageOps.equalize(im)
    return im

def pixel_stats(g):                                      # g: 2-D array in [0, 1]
    hist = np.histogram(g, bins=16, range=(0, 1))[0] / g.size
    border = np.concatenate([g[:, :20].ravel(), g[:, -20:].ravel(), g[:20].ravel(), g[-20:].ravel()]).mean()
    return [g.mean(), g.std(), laplace(g).var(), border, *hist]

def group_cv_accuracy(F, labels, groups, n_splits=5):
    k = int(min(n_splits, pd.Series(labels).value_counts().min()))
    accs = []
    for a, b in StratifiedGroupKFold(k, shuffle=True, random_state=SEED).split(F, labels, groups):
        m = RandomForestClassifier(200, random_state=SEED, n_jobs=-1).fit(F[a], labels[a])
        accs.append(m.score(F[b], labels[b]))
    return float(np.mean(accs))

if not cnn_ready:
    print('Image module skipped (RUN_CNN is False or the dataset folder was not found).')
    print(f'  expected dataset folder: {OUD_PATH}')
    print('  Download the Hamadan Ovarian Ultrasound Dataset (CC BY 4.0) and place its class folders there.')
else:
    # ---- 1) scan every file once
    recs, skipped = [], {'not_an_image': 0, 'no_class_folder': 0, 'unreadable': 0}
    for root, _, files in os.walk(OUD_PATH):
        for fn in sorted(files):
            fp = os.path.join(root, fn)
            if not fn.lower().endswith(IMG_EXT):
                skipped['not_an_image'] += 1; continue
            cls = oud_class(fp)
            if cls is None:
                skipped['no_class_folder'] += 1; continue
            try:
                with open(fp, 'rb') as f:
                    raw = f.read()
                with Image.open(io.BytesIO(raw)) as im:
                    im.load()
                    w, h = im.size
                    small = np.asarray(im.convert('L').resize((16, 16)), float)
                    pix = pixel_stats(np.asarray(to_model_gray(im), float) / 255.0)
                recs.append({'path': fp, 'cls': cls, 'md5': hashlib.md5(raw).hexdigest(),
                             'ahash': ''.join('1' if v > small.mean() else '0' for v in small.ravel()),
                             'w': w, 'h': h, 'pix': pix})
            except Exception:
                skipped['unreadable'] += 1
    if not recs:
        raise ValueError(f'No class-labelled images found under {OUD_PATH}. Folder names must contain "PCO", "Normal" or "Dominant".')
    scan = pd.DataFrame(recs)
    raw_counts = scan['cls'].value_counts().to_dict()

    # ---- 2) duplicates (exact = same bytes; near = same 16x16 average-hash, treated as one group)
    labels_per_md5 = scan.groupby('md5')['cls'].nunique()
    conflicting = set(labels_per_md5[labels_per_md5 > 1].index)
    n_exact_dups = int(len(scan) - scan['md5'].nunique())
    uniq = scan.drop_duplicates('md5')
    uniq = uniq[~uniq['md5'].isin(conflicting)].reset_index(drop=True)
    n_near_extra = int(len(uniq) - uniq['ahash'].nunique())
    uniq_counts = uniq['cls'].value_counts().to_dict()

    # ---- 3) shortcut audit for EVERY class pair: can size or simple pixel statistics alone separate the classes?
    usable = sorted([c for c, n in uniq_counts.items() if n >= MIN_IMAGES_PER_CLASS])
    pair_rows = []
    for a_, b_ in itertools.combinations(usable, 2):
        sub = uniq[uniq['cls'].isin([a_, b_])].reset_index(drop=True)
        lab, grp = sub['cls'].values, sub['ahash'].values
        size_f = np.c_[sub['w'].values, sub['h'].values, sub['w'].values / sub['h'].values].astype(float)
        s_acc = group_cv_accuracy(size_f, lab, grp)
        p_acc = group_cv_accuracy(np.array(sub['pix'].tolist()), lab, grp)
        base_acc = float(pd.Series(lab).value_counts(normalize=True).max())
        pair_rows.append({'pair': f'{a_} vs {b_}', 'classes': (a_, b_), 'n': int(len(sub)),
                          'size_only_acc': s_acc, 'pixel_stats_acc': p_acc, 'majority_baseline': base_acc,
                          'shortcut_acc': (max(s_acc, p_acc) if SIZE_SHORTCUT_COUNTS else p_acc),
                          'shortcut': bool((max(s_acc, p_acc) if SIZE_SHORTCUT_COUNTS else p_acc) >= SHORTCUT_THRESHOLD)})
    pairs_df = pd.DataFrame(pair_rows)

    # ---- 4) three-class task: all classes together
    class_list = [c for c in CNN_CLASSES if c in usable]
    if len(class_list) < 3 or CNN_POSITIVE not in class_list:
        raise ValueError(f'Need all 3 classes with >= {MIN_IMAGES_PER_CLASS} unique images: {uniq_counts}')
    pos_name = CNN_POSITIVE
    img_df = uniq[uniq['cls'].isin(class_list)].reset_index(drop=True)

    # shortcut baselines for the 3-class problem (no CNN)
    lab3, grp3 = img_df['cls'].values, img_df['ahash'].values
    size_f3 = np.c_[img_df['w'].values, img_df['h'].values, img_df['w'].values / img_df['h'].values].astype(float)
    s_acc3 = group_cv_accuracy(size_f3, lab3, grp3)
    p_acc3 = group_cv_accuracy(np.array(img_df['pix'].tolist()), lab3, grp3)
    base3  = float(pd.Series(lab3).value_counts(normalize=True).max())
    short3 = max(s_acc3, p_acc3) if SIZE_SHORTCUT_COUNTS else p_acc3
    # a pair of classes that image statistics alone can separate also invalidates the 3-class score for those classes,
    # even when the overall 3-class baseline is lower -> flag if the 3-class baseline OR any pair reaches the threshold
    shortcut_pairs = [r['pair'] for r in pair_rows if r['shortcut'] and set(r['classes']) <= set(class_list)]
    shortcut_any = bool(short3 >= SHORTCUT_THRESHOLD or shortcut_pairs)

    image_audit = {
        'dataset': 'Hamadan Ovarian Ultrasound Dataset (OUD)',
        'files_scanned': int(len(scan) + sum(skipped.values())), 'skipped': skipped,
        'class_counts_raw': {k: int(v) for k, v in raw_counts.items()},
        'exact_duplicate_files': n_exact_dups, 'near_duplicate_extra_images': n_near_extra,
        'same_image_in_both_classes': int(len(conflicting)),
        'class_counts_unique': {k: int(v) for k, v in uniq_counts.items()},
        'pairs': {r['pair']: {'n': r['n'], 'size_only_acc': r['size_only_acc'], 'pixel_stats_acc': r['pixel_stats_acc'],
                              'majority_baseline': r['majority_baseline'], 'shortcut': r['shortcut']} for r in pair_rows},
        'selected_pair': ' / '.join(class_list), 'positive_class': pos_name, 'negative_class': None, 'class_list': class_list,
        'selected_counts': {c: int((img_df['cls'] == c).sum()) for c in class_list},
        'size_only_accuracy': float(s_acc3),
        'pixel_stats_accuracy_after_normalisation': float(p_acc3),
        'majority_baseline': base3,
        'shortcut_threshold': SHORTCUT_THRESHOLD, 'shortcut_detected': shortcut_any, 'shortcut_pairs': shortcut_pairs,
        'shortcut_baseline_accuracy': float(short3),
        'preprocessing': {'crop_square': CROP_SQUARE, 'equalize_intensity': EQUALIZE_INTENSITY, 'blur_to_low_res': NORMALIZE_RES, 'size_counts_as_shortcut': SIZE_SHORTCUT_COUNTS},
        'normalised_resolution': int(LOW_RES) if NORMALIZE_RES else None,
    }
    pairs_df.drop(columns=['classes']).to_csv(os.path.join(OUTPUT_PATH, 'csv', 'image_audit_pairs.csv'), index=False)
    pd.Series({k: (json.dumps(v) if isinstance(v, dict) else v) for k, v in image_audit.items()},
              name='value').to_csv(os.path.join(OUTPUT_PATH, 'csv', 'image_audit.csv'))

    print('Ultrasound dataset audit')
    print('-' * 70)
    print(f"  Files scanned: {image_audit['files_scanned']} | skipped: {skipped}")
    print(f"  Images per class (raw)   : {image_audit['class_counts_raw']}")
    print(f"  Exact duplicate files    : {n_exact_dups} | extra near-duplicates: {n_near_extra} | same image under 2 labels: {len(conflicting)}")
    print(f"  Images per class (unique): {image_audit['class_counts_unique']}")
    print(f'\n  Preprocessing seen by the CNN: crop_square={CROP_SQUARE}, equalize_intensity={EQUALIZE_INTENSITY}, blur={NORMALIZE_RES}')
    print('  Can size or simple pixel statistics (no CNN) separate the classes?   [grouped 5-fold]')
    print(f"  (shortcut decision uses {'size and pixel statistics' if SIZE_SHORTCUT_COUNTS else 'pixel statistics of the normalised image; raw size is reported only'})")
    print(pairs_df[['pair', 'n', 'size_only_acc', 'pixel_stats_acc', 'majority_baseline', 'shortcut']].round(3).to_string(index=False))
    print(f"\n  CNN task: 3-class {' / '.join(class_list)}  | size-only {s_acc3:.3f} | pixel-stats {p_acc3:.3f} | majority {base3:.3f}")
    if image_audit['shortcut_detected']:
        print(f"  >>> SHORTCUT DETECTED (>= {SHORTCUT_THRESHOLD}) in: {shortcut_pairs or ['3-class baseline']}. "
              "Classes in these pairs can be separated by image statistics alone, so the CNN score for them is NOT evidence of ovarian-feature learning. "
              "Rely on the clean pairs (see pairwise results below).")
    else:
        print('  No shortcut above threshold for the 3-class task or any pair. The CNN score is still image-level (no patient IDs) and not clinically validated.')


In [ ]:
# ============================================================
#  3-class CNN (Normal / PCO / Dominant_Follicle): frozen ResNet50 features + multinomial
#  logistic head, evaluated by GROUPED 5-fold cross-validation (every image predicted once,
#  near-duplicate groups never split). Includes a label-permutation check (chance ~ 1/3).
# ============================================================
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import balanced_accuracy_score

def grouped_cv_probe(F, F_flip, y, groups, C, n_splits, seed):
    """Out-of-fold class probabilities (n_images x n_classes). Train on original + h-flipped; test on originals."""
    n_cls = int(y.max()) + 1
    oof = np.zeros((len(y), n_cls)); fold_id = np.zeros(len(y), int)
    for k, (a, b) in enumerate(StratifiedGroupKFold(n_splits, shuffle=True, random_state=seed).split(F, y, groups)):
        Xa = np.vstack([F[a], F_flip[a]]); ya = np.r_[y[a], y[a]]
        sc = StandardScaler().fit(Xa)
        clf = LogisticRegression(C=C, class_weight='balanced', max_iter=3000).fit(sc.transform(Xa), ya)
        proba = clf.predict_proba(sc.transform(F[b]))
        oof[np.ix_(b, clf.classes_)] = proba; fold_id[b] = k
    return oof, fold_id

def fold_scaler_into_head(sc, clf):
    """softmax(coef @ ((x-mu)/sd) + b) == softmax(W^T x + b')  -> one Dense(n_classes, softmax) on raw pooled features."""
    coef = clf.coef_                                           # (C, F)
    w = (coef / sc.scale_).T.astype('float32')                 # (F, C)
    b = (clf.intercept_ - np.sum(coef * sc.mean_ / sc.scale_, axis=1)).astype('float32')
    return w, b

cnn_model, pcos_idx, cnn_metrics, resnet_preprocess = None, None, None, None

if not (cnn_ready and image_audit is not None):
    print('CNN skipped (image audit did not run).')
else:
    from tensorflow.keras.applications.resnet50 import ResNet50, preprocess_input as resnet_preprocess

    def load_cnn_image(path):
        with Image.open(path) as im:
            return np.asarray(to_model_gray(im).convert('RGB'), dtype='float32')

    base = ResNet50(weights=CNN_WEIGHTS, include_top=False, pooling='avg', input_shape=IMG_SIZE + (3,))
    base.trainable = False

    def extract_features(paths, flip=False, bs=32):
        out = []
        for i in range(0, len(paths), bs):
            batch = np.stack([load_cnn_image(p) for p in paths[i:i + bs]])
            if flip:
                batch = batch[:, :, ::-1, :]
            out.append(base.predict(resnet_preprocess(batch.copy()), verbose=0))
        return np.vstack(out)

    cls_names = image_audit['class_list']                      # e.g. ['Normal', 'PCO', 'Dominant_Follicle']
    n_cls     = len(cls_names)
    pos_name  = image_audit['positive_class']
    pcos_idx  = cls_names.index(pos_name)
    cls_to_i  = {c: i for i, c in enumerate(cls_names)}
    paths     = img_df['path'].tolist()
    y_img         = img_df['cls'].map(cls_to_i).values.astype(int)
    groups    = img_df['ahash'].values
    counts    = np.bincount(y_img, minlength=n_cls)
    print(f'Extracting frozen ResNet50 features for {len(paths)} images: ' +
          ', '.join(f'{c}: {n}' for c, n in zip(cls_names, counts)) + ' ...')
    F, F_flip = extract_features(paths), extract_features(paths, flip=True)

    n_folds = int(min(CNN_CV_FOLDS, counts.min()))
    oof, fold_id = grouped_cv_probe(F, F_flip, y_img, groups, CNN_LR_C, n_folds, SEED)
    pred = oof.argmax(1)

    # ---- label-permutation check (chance for 3 balanced classes ~ 0.33)
    rng = np.random.RandomState(SEED)
    perm_bal = []
    for _ in range(N_PERM):
        yp = rng.permutation(y_img)
        o_p, _ = grouped_cv_probe(F, F_flip, yp, groups, CNN_LR_C, n_folds, SEED)
        perm_bal.append(balanced_accuracy_score(yp, o_p.argmax(1)))

    # ---- bootstrap CIs
    bs_acc, bs_auc, bs_f1 = [], [], []
    for _ in range(BOOTSTRAP_N):
        ix = rng.randint(0, len(y_img), len(y_img))
        if len(np.unique(y_img[ix])) < n_cls: continue
        bs_acc.append(accuracy_score(y_img[ix], pred[ix]))
        bs_f1.append(f1_score(y_img[ix], pred[ix], average='macro', zero_division=0))
        bs_auc.append(roc_auc_score(y_img[ix], oof[ix], multi_class='ovr', average='macro'))
    cm = confusion_matrix(y_img, pred, labels=list(range(n_cls)))
    per_class_recall = {c: float(cm[i, i] / max(cm[i].sum(), 1)) for i, c in enumerate(cls_names)}
    pairwise = {}
    for i_, j_ in itertools.combinations(range(n_cls), 2):
        m_ = np.isin(y_img, [i_, j_])
        p_ = np.where(oof[m_, i_] >= oof[m_, j_], i_, j_)           # decide between the two classes only
        nm_ = f'{cls_names[i_]} vs {cls_names[j_]}'
        ref_ = image_audit['pairs'].get(nm_) or image_audit['pairs'].get(f'{cls_names[j_]} vs {cls_names[i_]}')
        pairwise[nm_] = {'accuracy': float(accuracy_score(y_img[m_], p_)),
                         'balanced_accuracy': float(balanced_accuracy_score(y_img[m_], p_)),
                         'pixel_stats_baseline': float(ref_['pixel_stats_acc']) if ref_ else None,
                         'pair_shortcut': bool(ref_['shortcut']) if ref_ else None}
    fold_accs = [accuracy_score(y_img[fold_id == k], pred[fold_id == k]) for k in range(n_folds)]
    shortcut_acc = image_audit['shortcut_baseline_accuracy']
    cv_acc = float(accuracy_score(y_img, pred))
    cnn_metrics = {
        'task': f'{n_cls}-class: ' + ' / '.join(cls_names), 'head': 'frozen ResNet50 + multinomial logistic head',
        'n_images': int(len(y_img)), 'class_counts': {c: int(n) for c, n in zip(cls_names, counts)}, 'cv_folds': n_folds,
        'cv_accuracy': cv_acc, 'cv_accuracy_ci95_low': float(np.percentile(bs_acc, 2.5)), 'cv_accuracy_ci95_high': float(np.percentile(bs_acc, 97.5)),
        'cv_balanced_accuracy': float(balanced_accuracy_score(y_img, pred)),
        'cv_macro_f1': float(f1_score(y_img, pred, average='macro', zero_division=0)),
        'cv_macro_f1_ci95_low': float(np.percentile(bs_f1, 2.5)), 'cv_macro_f1_ci95_high': float(np.percentile(bs_f1, 97.5)),
        'cv_auc': float(roc_auc_score(y_img, oof, multi_class='ovr', average='macro')),
        'cv_auc_ci95_low': float(np.percentile(bs_auc, 2.5)), 'cv_auc_ci95_high': float(np.percentile(bs_auc, 97.5)),
        'per_class_recall': per_class_recall,
        'pairwise_oof': pairwise,
        'fold_accuracy_std': float(np.std(fold_accs)),
        'majority_baseline': float(counts.max() / counts.sum()),
        'permuted_label_balanced_accuracy_mean': float(np.mean(perm_bal)),
        'shortcut_baseline_accuracy': float(shortcut_acc),
        'shortcut_detected': bool(image_audit['shortcut_detected']),
        'beats_simple_pixel_stats': bool(cv_acc > shortcut_acc),
        'interpretable_as_diagnostic': bool(not image_audit['shortcut_detected']),
    }
    print('\nCNN grouped cross-validation (out-of-fold, every image predicted once)')
    print('-' * 62)
    for k_, v_ in cnn_metrics.items():
        print(f'  {k_:<40}: {v_:.4f}' if isinstance(v_, float) else f'  {k_:<40}: {v_}')
    print(classification_report(y_img, pred, target_names=cls_names, zero_division=0))
    print('  Pairwise accuracy of the 3-class model (out-of-fold, two classes at a time; pixel-stats baseline in brackets):')
    for nm_, d_ in pairwise.items():
        base_s = f"{d_['pixel_stats_baseline']:.3f}" if d_['pixel_stats_baseline'] is not None else 'n/a'
        print(f"    {nm_:<32}: acc {d_['accuracy']:.3f} | bal.acc {d_['balanced_accuracy']:.3f} [{base_s}]"
              + ('   <-- shortcut pair: not evidence' if d_['pair_shortcut'] else ''))
    chance = 1.0 / n_cls
    if cnn_metrics['permuted_label_balanced_accuracy_mean'] > chance + 0.12:
        print('  >>> Shuffled labels still score above chance: investigate leakage before trusting this result.')
    if cnn_metrics['shortcut_detected']:
        print(f"  >>> Shortcut in {image_audit['shortcut_pairs'] or ['3-class baseline']}: the overall 3-class score is NOT evidence of ovarian-feature learning. Use the clean pairs only.")
    elif not cnn_metrics['beats_simple_pixel_stats']:
        print(f'  >>> The CNN does not beat simple pixel statistics ({shortcut_acc:.3f}): no evidence it learned more than brightness/texture.')

    pd.DataFrame({'path': paths, 'class': img_df['cls'].values, 'fold': fold_id,
                  **{f'p_{c}': oof[:, i] for i, c in enumerate(cls_names)}}
                 ).to_csv(os.path.join(OUTPUT_PATH, 'csv', 'cnn_oof_predictions.csv'), index=False)

    fig, ax = plt.subplots(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax, xticklabels=cls_names, yticklabels=cls_names)
    ax.set_title('CNN out-of-fold confusion matrix (3-class)'); ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
    plt.xticks(rotation=20, ha='right'); plt.tight_layout(); save_fig('Fig5_CNN_confusion_matrix.png'); plt.show()

    # ---- final model on all images; scaler folded into one Dense(n_cls, softmax)
    Xa = np.vstack([F, F_flip]); ya = np.r_[y_img, y_img]
    sc_f = StandardScaler().fit(Xa)
    clf_f = LogisticRegression(C=CNN_LR_C, class_weight='balanced', max_iter=3000).fit(sc_f.transform(Xa), ya)
    head = tf.keras.layers.Dense(n_cls, activation='softmax', name='oud_head')
    cnn_model = tf.keras.Model(base.input, head(base.output))
    head.set_weights(list(fold_scaler_into_head(sc_f, clf_f)))
    chk = np.stack([load_cnn_image(p) for p in paths[:8]])
    diff = float(np.max(np.abs(cnn_model.predict(resnet_preprocess(chk.copy()), verbose=0)
                               - clf_f.predict_proba(sc_f.transform(F[:8])))))
    print(f'\nFinal model self-check: Keras head vs sklearn head max |diff| = {diff:.2e}')
    cnn_model.save(os.path.join(OUTPUT_PATH, 'models', 'cnn_model.keras'))

    # ---- Grad-CAM for the PCO output + border-attention check
    def gradcam_heatmap(model, img_batch, class_idx, layer_name='conv5_block3_out'):
        grad_model = tf.keras.models.Model(model.inputs, [model.get_layer(layer_name).output, model.output])
        with tf.GradientTape() as tape:
            conv_out, preds = grad_model(img_batch)
            score = preds[:, class_idx]
        grads = tape.gradient(score, conv_out)
        w_ = tf.reduce_mean(grads, axis=(1, 2))
        cam = tf.nn.relu(tf.reduce_sum(conv_out * w_[:, None, None, :], axis=-1)[0])
        return (cam / (tf.reduce_max(cam) + 1e-8)).numpy()

    def border_attention(cam_up, frac):
        m = int(round(frac * cam_up.shape[0]))
        inner = cam_up[m:-m, m:-m].sum()
        return float(1.0 - inner / (cam_up.sum() + 1e-8))

    border_vals = []
    for pos_i in np.where(y_img == pcos_idx)[0][:N_GRADCAM]:
        arr = load_cnn_image(paths[int(pos_i)])
        cam = gradcam_heatmap(cnn_model, resnet_preprocess(arr[None].copy()), pcos_idx)
        cam_up = tf.image.resize(cam[..., None], IMG_SIZE).numpy()[..., 0]
        border_vals.append(border_attention(cam_up, BORDER_FRAC))
        if len(border_vals) == 1:
            first_arr, first_cam = arr, cam_up
    if border_vals:
        expected = 1.0 - (1.0 - 2 * BORDER_FRAC) ** 2
        cnn_metrics['gradcam_border_attention_mean'] = float(np.mean(border_vals))
        cnn_metrics['gradcam_border_area_fraction'] = float(expected)
        print(f"Grad-CAM attention in the outer {BORDER_FRAC:.0%} margin: {np.mean(border_vals):.3f} "
              f"(uniform attention would give {expected:.3f}; n={len(border_vals)} {pos_name} images)")
        fig, axes = plt.subplots(1, 3, figsize=(13, 4.6))
        axes[0].imshow(first_arr.astype('uint8')); axes[0].set_title('(a) Normalised ultrasound')
        axes[1].imshow(first_cam, cmap='jet');      axes[1].set_title(f'(b) Grad-CAM ({pos_name} output)')
        axes[2].imshow(first_arr.astype('uint8')); axes[2].imshow(first_cam, cmap='jet', alpha=0.4)
        axes[2].set_title('(c) Overlay')
        for a_ in axes: a_.axis('off')
        fig.suptitle('Figure 6. Grad-CAM, ResNet50 (illustrative; model fitted on all images)', y=1.02)
        plt.tight_layout(); save_fig('Fig6_GradCAM.png'); plt.show()

### Module 5b — Fine-tuned ResNet50 with augmentation (optional)
Same grouped folds as the frozen probe above, so the two are directly comparable. Epochs and learning rates are fixed in the config, so nothing is chosen on the test folds. Every class pair is compared with a paired bootstrap; a gain on a shortcut pair (Normal vs the others) is **not** evidence of ovarian-feature learning. Needs a GPU; set `RUN_FINETUNE = False` to skip.


In [ ]:
# ============================================================
#  Optional: FINE-TUNED ResNet50 with augmentation, same grouped 5-fold splits as the frozen probe.
#  Nothing is selected on the test fold: epochs and learning rates are fixed in the config.
#  Output: side-by-side comparison with the frozen-feature result, and a paired bootstrap of the
#  difference for every class pair (shortcut pairs are marked: a gain there is not evidence).
# ============================================================
import time
ft_metrics = None

def _pair_preds(oof, i_, j_, y):
    m_ = np.where(np.isin(y, [i_, j_]))[0]
    return m_, np.where(oof[m_, i_] >= oof[m_, j_], i_, j_)

def ft_summary(oof, y, names):
    oof = np.asarray(oof, 'float64'); oof = oof / oof.sum(1, keepdims=True)
    pred = oof.argmax(1); cm_ = confusion_matrix(y, pred, labels=list(range(len(names))))
    d = {'accuracy': float(accuracy_score(y, pred)), 'balanced_accuracy': float(balanced_accuracy_score(y, pred)),
         'macro_f1': float(f1_score(y, pred, average='macro', zero_division=0)),
         'auc': float(roc_auc_score(y, oof, multi_class='ovr', average='macro')),
         'per_class_recall': {c: float(cm_[i, i] / max(cm_[i].sum(), 1)) for i, c in enumerate(names)}, 'pairwise': {}}
    for i_, j_ in itertools.combinations(range(len(names)), 2):
        m_, p_ = _pair_preds(oof, i_, j_, y)
        d['pairwise'][f'{names[i_]} vs {names[j_]}'] = {'accuracy': float(accuracy_score(y[m_], p_)),
                                                         'balanced_accuracy': float(balanced_accuracy_score(y[m_], p_))}
    return d, cm_

def paired_pair_diff(y, oof_a, oof_b, i_, j_, n_boot, seed):
    """accuracy(b) - accuracy(a) on one class pair, with a paired bootstrap 95% CI."""
    m_, pa = _pair_preds(oof_a, i_, j_, y); _, pb = _pair_preds(oof_b, i_, j_, y)
    ca, cb = (pa == y[m_]).astype(float), (pb == y[m_]).astype(float)
    rng_ = np.random.RandomState(seed); diffs = []
    for _ in range(n_boot):
        ix = rng_.randint(0, len(m_), len(m_)); diffs.append(cb[ix].mean() - ca[ix].mean())
    return float(cb.mean() - ca.mean()), float(np.percentile(diffs, 2.5)), float(np.percentile(diffs, 97.5))

if not (RUN_FINETUNE and cnn_metrics is not None):
    print('Fine-tuning skipped (RUN_FINETUNE is False or the frozen CNN did not run).')
elif not tf.config.list_physical_devices('GPU') and not FT_ALLOW_CPU:
    print('Fine-tuning skipped: no GPU found. In Colab use Runtime > Change runtime type > T4 GPU,')
    print('or set FT_ALLOW_CPU = True in the config (will be very slow).')
else:
    from sklearn.utils.class_weight import compute_class_weight
    L = tf.keras.layers
    X_img = np.stack([load_cnn_image(p) for p in paths]).astype('float32')       # same preprocessing as the audit / frozen probe

    def finetune_fold(Xtr, ytr, Xte, seed):
        """Train on one fold's training images; return softmax probabilities for Xte (flip test-time averaging)."""
        tf.keras.backend.clear_session(); tf.keras.utils.set_random_seed(seed)
        base_ = tf.keras.applications.ResNet50(weights=CNN_WEIGHTS, include_top=False, input_shape=IMG_SIZE + (3,))
        inp = tf.keras.Input(IMG_SIZE + (3,))
        x = L.RandomFlip('horizontal')(inp)                 # augmentation layers are active only during fit()
        x = L.RandomRotation(FT_ROTATION)(x)
        x = L.RandomZoom(FT_ZOOM)(x)
        x = L.RandomContrast(FT_CONTRAST)(x)
        x = L.Lambda(lambda t: tf.keras.applications.resnet50.preprocess_input(t))(x)
        x = base_(x, training=False)                        # BatchNorm stays in inference mode (small dataset)
        x = L.GlobalAveragePooling2D()(x)
        x = L.Dropout(0.3)(x)
        out = L.Dense(n_cls, activation='softmax')(x)
        model = tf.keras.Model(inp, out)
        cw = dict(enumerate(compute_class_weight('balanced', classes=np.arange(n_cls), y=ytr)))
        # phase 1: train the new head only
        base_.trainable = False
        model.compile(tf.keras.optimizers.Adam(1e-3), 'sparse_categorical_crossentropy')
        model.fit(Xtr, ytr, epochs=FT_HEAD_EPOCHS, batch_size=FT_BATCH, class_weight=cw, verbose=0)
        # phase 2: unfreeze the last ResNet stage (BatchNorm layers stay frozen), small learning rate
        names_ = [l.name for l in base_.layers]; k0 = names_.index(FT_UNFREEZE_FROM)
        for l_ in base_.layers:
            l_.trainable = False
        for l_ in base_.layers[k0:]:
            if not isinstance(l_, L.BatchNormalization):
                l_.trainable = True
        model.compile(tf.keras.optimizers.Adam(FT_LR), 'sparse_categorical_crossentropy')
        model.fit(Xtr, ytr, epochs=FT_EPOCHS, batch_size=FT_BATCH, class_weight=cw, verbose=0)
        p = 0.5 * (model.predict(Xte, verbose=0) + model.predict(Xte[:, :, ::-1, :].copy(), verbose=0))
        del model, base_; tf.keras.backend.clear_session()
        return p

    # ---- run grouped CV (identical folds to the frozen probe: fold_id comes from the previous cell)
    oof_ft = np.zeros((len(y_img), n_cls)); t0 = time.time()
    print(f'Fine-tuning ResNet50 ({FT_HEAD_EPOCHS} head + {FT_EPOCHS} fine-tune epochs per fold, {n_folds} folds) ...')
    for k_ in range(n_folds):
        tr, te = np.where(fold_id != k_)[0], np.where(fold_id == k_)[0]
        oof_ft[te] = finetune_fold(X_img[tr], y_img[tr], X_img[te], SEED + k_)
        print(f'  fold {k_ + 1}/{n_folds}: acc {accuracy_score(y_img[te], oof_ft[te].argmax(1)):.3f}  ({time.time() - t0:.0f}s elapsed)')

    sum_frozen, _ = ft_summary(oof, y_img, cls_names)
    sum_ft, cm_ft = ft_summary(oof_ft, y_img, cls_names)

    rows_ = []
    for tag_, s_ in [('Frozen ResNet50 + logistic head', sum_frozen), ('Fine-tuned ResNet50 + augmentation', sum_ft)]:
        r_ = {'model': tag_, 'accuracy': s_['accuracy'], 'balanced_acc': s_['balanced_accuracy'], 'macro_F1': s_['macro_f1'], 'AUC': s_['auc']}
        for nm_, v_ in s_['pairwise'].items():
            r_[nm_ + (' (shortcut)' if cnn_metrics['pairwise_oof'][nm_]['pair_shortcut'] else '')] = v_['accuracy']
        rows_.append(r_)
    cmp_df = pd.DataFrame(rows_).set_index('model')
    print('\nOut-of-fold comparison (same grouped folds, every image predicted once)')
    print('-' * 100)
    print(cmp_df.round(3).T.to_string())
    print('\nPixel-statistics baseline per pair (no CNN): ' + ' | '.join(
        f"{nm_} {d_['pixel_stats_baseline']:.3f}" for nm_, d_ in cnn_metrics['pairwise_oof'].items()))

    diffs_ = {}
    print('\nFine-tuned minus frozen, accuracy per pair (paired bootstrap 95% CI)')
    for i_, j_ in itertools.combinations(range(n_cls), 2):
        nm_ = f'{cls_names[i_]} vs {cls_names[j_]}'
        d_, lo_, hi_ = paired_pair_diff(y_img, oof, oof_ft, i_, j_, BOOTSTRAP_N, SEED)
        shortcut_ = cnn_metrics['pairwise_oof'][nm_]['pair_shortcut']
        diffs_[nm_] = {'diff': d_, 'ci95_low': lo_, 'ci95_high': hi_, 'pair_shortcut': shortcut_}
        verdict_ = ('shortcut pair: not evidence' if shortcut_ else
                    'reliable gain' if lo_ > 0 else 'reliable loss' if hi_ < 0 else 'no reliable difference')
        print(f'  {nm_:<32}: {d_:+.3f}  [{lo_:+.3f}, {hi_:+.3f}]   {verdict_}')

    print('\nFine-tuned per-class report (3-class, out-of-fold)')
    print(classification_report(y_img, oof_ft.argmax(1), target_names=cls_names, zero_division=0))
    fig, ax = plt.subplots(figsize=(6, 5))
    sns.heatmap(cm_ft, annot=True, fmt='d', cmap='Blues', ax=ax, xticklabels=cls_names, yticklabels=cls_names)
    ax.set_title('Fine-tuned CNN out-of-fold confusion matrix'); ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
    plt.xticks(rotation=20, ha='right'); plt.tight_layout(); save_fig('Fig5b_CNN_finetuned_confusion_matrix.png'); plt.show()

    ft_metrics = {'method': f'ResNet50 fine-tuned (last stage, {FT_HEAD_EPOCHS}+{FT_EPOCHS} epochs, fixed) with augmentation, flip TTA',
                  'folds': int(n_folds), 'frozen': sum_frozen, 'finetuned': sum_ft, 'finetuned_minus_frozen_by_pair': diffs_,
                  'note': 'Image-level grouped CV (no patient IDs). Gains on shortcut pairs are not evidence of ovarian-feature learning.'}
    cmp_df.to_csv(os.path.join(OUTPUT_PATH, 'csv', 'cnn_finetune_comparison.csv'))
    pd.DataFrame({'path': paths, 'class': img_df['cls'].values, 'fold': fold_id,
                  **{f'p_{c}': oof_ft[:, i] for i, c in enumerate(cls_names)}}
                 ).to_csv(os.path.join(OUTPUT_PATH, 'csv', 'cnn_finetune_oof_predictions.csv'), index=False)


## Module 6 — GenAI advisory (Groq)
The API key is **never stored in the notebook**. In Colab: left sidebar → 🔑 Secrets → add `GROQ_API_KEY` and enable notebook access. Locally: set the `GROQ_API_KEY` environment variable. Revoke the key that was pasted into the earlier notebook.
The prompt contains the predicted subtype, the model confidence and the top SHAP drivers for *that patient*, and asks for educational guidance only (no doses, no diagnosis).

In [ ]:
# ============================================================
#   GenAI advisory driven by subtype + SHAP drivers
# ============================================================
def get_groq_client():
    key = None
    try:
        from google.colab import userdata
        key = userdata.get('GROQ_API_KEY')
    except Exception:
        pass
    key = key or os.environ.get('GROQ_API_KEY')
    if not key:
        return None
    try:
        from groq import Groq
    except ImportError:
        print('groq package not installed (pip install groq) - advisory skipped.')
        return None
    return Groq(api_key=key)

def build_advisory_prompt(res):
    drivers = '\n'.join(f"- {d['feature']} = {d['value']:.2f} ({d['direction']} the likelihood of this subtype)"
                        for d in res['drivers'])
    extra = ''
    if res.get('cnn_pcos_probability') is not None and not (image_audit and image_audit['shortcut_detected']):
        extra = f"\nUltrasound CNN estimate of PCOS-pattern probability: {res['cnn_pcos_probability']:.0%}."
    return (
        'You are a health-education assistant. A machine-learning model trained on a small clinical dataset '
        'with proxy-rule labels estimated a PCOS subtype for a patient. This is NOT a diagnosis.\n\n'
        f"Estimated subtype: {res['subtype']} (model confidence {res['confidence']:.0%}).\n"
        f'Most influential factors for this patient:\n{drivers}{extra}\n\n'
        'Write exactly 5 short bullet points of general educational guidance tailored to this subtype and these factors: '
        '(1) diet, (2) physical activity, (3) sleep/stress, (4) tests or follow-up worth discussing with a doctor, '
        '(5) when to seek medical care. Mention medicines or supplements only as topics to ask a doctor about. '
        'Give no doses, do not claim certainty, and use only the values given above. '
        'Do not attribute hormone levels (e.g. cortisol, DHEA-S, insulin) to the patient: those were not measured and the label is a proxy rule. '
        'Describe the given values neutrally: do not call any of them low, high or deficient.'
    )

def get_advisory(client, res):
    try:
        r = client.chat.completions.create(
            model=GROQ_MODEL, temperature=0.3,
            messages=[{'role': 'user', 'content': build_advisory_prompt(res)}])
        return r.choices[0].message.content.strip() + f'\n\n_{ADVISORY_DISCLAIMER}_'
    except Exception as e:
        return f'[advisory unavailable: {type(e).__name__}: {e}]'

groq_client = get_groq_client() if RUN_GENAI else None
if RUN_GENAI and groq_client is None:
    print('GROQ_API_KEY not found (Colab Secrets or environment) - advisory demo skipped; predictions still shown.')

demo_records = []
print('Patient-level prediction demo (final model; patients taken from the labelled cohort)')
print('=' * 70)
for s in class_names:
    idxs = np.where(df_model['Subtype_Name'].values == s)[0][:N_DEMO_PER_SUBTYPE]
    for i in idxs:
        patient = X_final.iloc[i].to_dict()
        res = predict_patient(patient)
        rec = {'rule_label': s, 'prediction': res}
        print(f'\nRule label: {s}  ->  model: {res["subtype"]} ({res["confidence"]:.0%})')
        for d in res['drivers']:
            print(f"   {d['feature']:<20} = {d['value']:8.2f}  ({d['direction']} likelihood)")
        if groq_client is not None:
            rec['advisory'] = get_advisory(groq_client, res)
            print('\n' + rec['advisory'])
        demo_records.append(rec)

with open(os.path.join(OUTPUT_PATH, 'csv', 'genai_demo.json'), 'w') as f:
    json.dump(demo_records, f, indent=2, default=str)

## Module 7 — Comparison table, results summary and paper tables
Everything below reads from variables. `paper_tables.md` in `outputs/csv` contains the markdown tables used in the paper, generated from this run.

In [ ]:
# ============================================================
#  Comparison table vs existing papers
#  Reported accuracy of other papers is left blank on purpose: copy it from each paper.
# ============================================================
ens_full  = tables[FINAL_FEATURE_SET].loc['Ensemble']
blind_acc = (tables['blind_features'].loc['Ensemble', 'Accuracy'] if 'blind_features' in tables else np.nan)

rows = [
    ['Bharati et al. 2020',  'Binary PCOS / no-PCOS',        'Clinical (tabular)',  'No', 'No', 'No', np.nan],
    ['Denny et al. 2021',    'Binary PCOS / no-PCOS',        'Clinical (Kerala)',   'No', 'Yes (explainable ML)', 'No', np.nan],
    ['Modi & Kumar 2024',    'Binary PCOS / no-PCOS',        'Clinical (tabular)',  'No', 'No', 'No', np.nan],
    ['Chitra et al. 2023',   'Binary (ultrasound CNN)',      'Ultrasound images',   'No', 'No', 'No', np.nan],
    ['F-Net 2024',           'Binary + follicle detection',  'Ultrasound images',   'No', 'No', 'No', np.nan],
    ['This work (ensemble, full features)',  f'{n_classes}-class subtype (proxy-rule labels)',
        'Clinical', 'Yes (proxy)', 'Yes (SHAP)', 'Yes', float(ens_full['Accuracy'])],
    ['This work (ensemble, blind features)', f'{n_classes}-class subtype (proxy-rule labels)',
        'Clinical, rule columns removed', 'Yes (proxy)', 'Yes (SHAP)', 'Yes', float(blind_acc)],
]
if cnn_metrics:
    note = 'shortcut-confounded: not comparable' if cnn_metrics['shortcut_detected'] else 'image-level grouped CV, no patient IDs'
    rows.append([f"This work (ResNet50 features, {cnn_metrics['task']})", f'3-class ultrasound ({note})', 'Ultrasound images',
                 'No', 'Grad-CAM', 'No', np.nan if cnn_metrics['shortcut_detected'] else cnn_metrics['cv_accuracy']])
if ft_metrics:
    for _nm, _d in cnn_metrics['pairwise_oof'].items():
        if not _d['pair_shortcut']:
            rows.append([f'This work (ResNet50 frozen, clean pair {_nm})', 'binary ultrasound pair (image-level grouped CV, no patient IDs)',
                         'Ultrasound images', 'No', 'Grad-CAM', 'No', _d['accuracy']])
            rows.append([f'This work (ResNet50 fine-tuned, clean pair {_nm})', 'binary ultrasound pair (image-level grouped CV, no patient IDs)',
                         'Ultrasound images', 'No', 'No', 'No', ft_metrics['finetuned']['pairwise'][_nm]['accuracy']])
lit = pd.DataFrame(rows, columns=['Paper', 'Task', 'Modality', 'Subtype_classes', 'Explainability',
                                  'GenAI_advisory', 'Accuracy_reported'])
lit.to_csv(os.path.join(OUTPUT_PATH, 'csv', 'comparison_vs_literature.csv'), index=False)
print('Accuracies of other papers must be filled in from the papers themselves (left blank here).')
print('Task difficulty and label type differ, so raw accuracy is not directly comparable.\n')
print(lit.to_string(index=False))


In [ ]:
# ============================================================
#  Results summary (read from variables) + paper_tables.md
# ============================================================
def _clean(o):
    if isinstance(o, (np.integer,)):  return int(o)
    if isinstance(o, (np.floating,)): return float(o)
    if isinstance(o, (np.bool_,)):    return bool(o)
    return str(o)

n_adv_ok = sum(1 for r in demo_records if r.get('advisory') and not r['advisory'].startswith('[advisory unavailable'))
summary = {
    'dataset': {
        'all_patients': int(len(df_raw)), 'pcos_positive': int(len(cohort)),
        'excluded_missing_rule_input': int(n_unlabelable), 'unclassified_by_rules': n_unclassified,
        'used_for_modelling': int(len(df_model)), 'candidate_features': int(X_all.shape[1]),
        'dropped_small_subtypes': dropped_small,
    },
    'labelling': {'method': 'clinical-proxy rule cascade', 'rules': RULES,
                  'distribution': dist_df.to_dict(orient='index'),
                  'features_differing_between_subtypes_FDR05': n_sig,
                  'features_tested_kruskal': int(len(kw)), 'homa_ir': homa_status},
    'majority_baseline_accuracy': majority_baseline,
    'experiments': {tag: {'n_folds': results[tag]['n_splits'], 'n_candidate_features': len(EXPERIMENTS[tag]),
                          'metrics': tables[tag].round(4).to_dict(orient='index')} for tag in results},
    'top5_shap_per_subtype': {n: top5_df[top5_df['Subtype'] == n]['Feature'].tolist() for n in class_names},
    'image_audit': image_audit,
    'cnn': cnn_metrics,
    'cnn_finetune': ft_metrics,
    'genai': {'model': GROQ_MODEL, 'ran': bool(n_adv_ok > 0), 'advisories_generated': int(n_adv_ok),
              'demo_patients': len(demo_records)},
}
with open(os.path.join(OUTPUT_PATH, 'csv', 'results_summary.json'), 'w') as f:
    json.dump(summary, f, indent=2, default=_clean)

def md_table(df, fmt='{:.3f}'):
    cols = [str(df.index.name or '')] + [str(c) for c in df.columns]
    lines = ['| ' + ' | '.join(cols) + ' |', '|' + '---|' * len(cols)]
    for idx, row in df.iterrows():
        cells_ = [str(idx)] + [('{:.0f}'.format(v) if c == 'support' else fmt.format(v)) if isinstance(v, (float, np.floating)) else str(v)
                              for c, v in zip(df.columns, row.values)]
        lines.append('| ' + ' | '.join(cells_) + ' |')
    return '\n'.join(lines)

parts = ['# Paper tables (generated by the notebook; all values computed in this run)\n']
parts.append('## Label distribution vs plan reference ranges\n' + md_table(dist_df, '{:.1f}') + '\n')
parts.append(f'Cohort: {len(df_raw)} patients, {len(cohort)} PCOS-positive, {n_unlabelable} excluded (missing rule input), '
             f'{n_unclassified} unclassified, {len(df_model)} used. Dropped small subtypes: {dropped_small}. '
             f'Majority-class baseline accuracy: {majority_baseline:.3f}.\n')
for tag in results:
    t = tables[tag]
    parts.append(f'## {tag} ({len(EXPERIMENTS[tag])} candidate features, {results[tag]["n_splits"]}-fold out-of-fold)\n'
                 + md_table(t[['Accuracy', 'Macro_F1', 'Weighted_F1', 'Macro_F1_CI95_low', 'Macro_F1_CI95_high']]) + '\n')
    parts.append(f'### Ensemble per-subtype report ({tag})\n'
                 + md_table(reports[tag].loc[class_names + ['macro avg', 'weighted avg']].round(3)) + '\n')
if len(kw):
    parts.append(f'## Kruskal-Wallis on {len(kw)} non-rule columns ({n_sig} significant at FDR < 0.05)\n'
                 + md_table(kw.head(10).set_index('Feature'), '{:.4f}') + '\n')
parts.append('## Top-5 SHAP features per subtype\n'
             + md_table(pd.DataFrame({n: [', '.join(top5_df[top5_df['Subtype'] == n]['Feature'])] for n in class_names}).T.rename(columns={0: 'Top 5'})) + '\n')
if image_audit:
    a = image_audit
    pr = pd.DataFrame(a['pairs']).T[['n', 'size_only_acc', 'pixel_stats_acc', 'majority_baseline', 'shortcut']]
    pr.index.name = 'class pair'
    parts.append('## Image dataset audit (' + a['dataset'] + ')\n'
                 + f"Raw images per class: {a['class_counts_raw']}; exact duplicate files: {a['exact_duplicate_files']}; "
                   f"extra near-duplicates: {a['near_duplicate_extra_images']}; same image under two labels: {a['same_image_in_both_classes']}.\n\n"
                 + md_table(pr.astype({'n': int, 'size_only_acc': float, 'pixel_stats_acc': float, 'majority_baseline': float})) + '\n')
    parts.append(f"Selected CNN task: {a['selected_pair']} {a['selected_counts']}; shortcut detected: {a['shortcut_detected']}.\n")
if cnn_metrics:
    parts.append('## CNN (grouped cross-validation)\n' + md_table(pd.DataFrame({'value': {k: (f'{v:.4f}' if isinstance(v, float) else v)
                                                                             for k, v in cnn_metrics.items()}})) + '\n')
parts.append('## Comparison table\n' + md_table(lit.set_index('Paper').fillna('')) + '\n')
with open(os.path.join(OUTPUT_PATH, 'csv', 'paper_tables.md'), 'w') as f:
    f.write('\n'.join(parts))

print('FINAL RESULTS SUMMARY (all values computed in this run)')
print('=' * 70)
ds = summary['dataset']
print(f"Patients: {ds['all_patients']} total | {ds['pcos_positive']} PCOS+ | {ds['used_for_modelling']} labelled and used")
print('Subtype distribution:')
for s in class_names:
    r = dist_df.loc[s]
    print(f"  {s:<20}: {int(r['Count']):4d} ({r['Percent']:.1f}%)")
print(f'Majority-class baseline accuracy: {majority_baseline:.4f}')
for tag in results:
    t = tables[tag]
    print(f'\n[{tag}] {results[tag]["n_splits"]}-fold out-of-fold')
    print(t[['Accuracy', 'Macro_F1', 'Weighted_F1']].round(4).to_string())
    lo, hi = t.loc['Ensemble', 'Macro_F1_CI95_low'], t.loc['Ensemble', 'Macro_F1_CI95_high']
    print(f'  Ensemble macro-F1 95% CI: [{lo:.3f}, {hi:.3f}]')
if image_audit:
    print(f"\nImage audit ({image_audit['dataset']}): task {image_audit['selected_pair']} | size-only {image_audit['size_only_accuracy']:.3f} "
          f"| pixel-stats {image_audit['pixel_stats_accuracy_after_normalisation']:.3f} | exact duplicate files {image_audit['exact_duplicate_files']}")
if cnn_metrics:
    verdict = (f"NOT interpretable as diagnostic overall (shortcut pairs: {image_audit['shortcut_pairs']}; see pairwise_oof)" if cnn_metrics['shortcut_detected']
               else 'no shortcut above threshold (image-level, unvalidated)')
    print(f"CNN ({cnn_metrics['task']}, {cnn_metrics['cv_folds']}-fold grouped CV): accuracy {cnn_metrics['cv_accuracy']:.4f} "
          f"[{cnn_metrics['cv_accuracy_ci95_low']:.3f}, {cnn_metrics['cv_accuracy_ci95_high']:.3f}], AUC {cnn_metrics['cv_auc']:.4f}, "
          f"balanced acc {cnn_metrics['cv_balanced_accuracy']:.4f}, permuted-label bal. acc {cnn_metrics['permuted_label_balanced_accuracy_mean']:.3f} -> {verdict}")
if ft_metrics:
    for _nm, _d in ft_metrics['finetuned_minus_frozen_by_pair'].items():
        _tag = 'shortcut pair: not evidence' if _d['pair_shortcut'] else 'clean pair'
        print(f"Fine-tuned minus frozen, {_nm}: {_d['diff']:+.3f} [{_d['ci95_low']:+.3f}, {_d['ci95_high']:+.3f}] ({_tag})")
print(f'\nGenAI advisories generated: {n_adv_ok} of {len(demo_records)} demo patients')
print('\nFiles written to', OUTPUT_PATH)
